In [0]:
%run "../libs/notebook_init"

In [ ]:
# =============================================================================
# Step log init (STATUS_RUNNING)
# =============================================================================

nb = Utils.get_notebook_context(dbutils)
notebook_folder = nb["notebook_folder"]
notebook_name   = nb["notebook_name"]

step_log_id       = str(uuid.uuid4())
pipeline_run_id   = PIPELINE_RUN_ID
step_sequence     = 1
layer             = "bronze"
target_table      = None
status            = STATUS_RUNNING
started_timestamp = datetime.now(timezone.utc)
rows_read         = 0
rows_written      = 0
error_message     = None

pipeline_step_log_upsert(
    spark, step_log_id, pipeline_run_id, step_sequence,
    notebook_folder, notebook_name, status, started_timestamp,
    layer, target_table
)


In [ ]:
SUBPATH = "productdata"                       # case must match volume
SOURCE_PATH    = f"{RAW_FILES}{SUBPATH}"


try:
    Utils.move_all_files(
        dbutils,
        source_path = f"{SOURCE_PATH}/archive",
        target_path = SOURCE_PATH,
        create_target = False,
        skip_dirs = True,
        use_date_partition = False
    )

    SUBPATH = "arancione"                       # case must match volume
    SOURCE_PATH    = f"{RAW_FILES}{SUBPATH}"
    Utils.move_all_files(
        dbutils,
        source_path = f"{SOURCE_PATH}/archive",
        target_path = SOURCE_PATH,
        create_target = False,
        skip_dirs = True,
        use_date_partition = False
    )

    SUBPATH = "celeste"                       # case must match volume
    SOURCE_PATH    = f"{RAW_FILES}{SUBPATH}"
    Utils.move_all_files(
        dbutils,
        source_path = f"{SOURCE_PATH}/archive",
        target_path = SOURCE_PATH,
        create_target = False,
        skip_dirs = True,
        use_date_partition = False
    )

    SUBPATH = "verde"                       # case must match volume
    SOURCE_PATH    = f"{RAW_FILES}{SUBPATH}"
    Utils.move_all_files(
        dbutils,
        source_path = f"{SOURCE_PATH}/archive",
        target_path = SOURCE_PATH,
        create_target = False,
        skip_dirs = True,
        use_date_partition = False
    )

    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_SUCCEEDED


    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )


except Exception as e:
    err = Utils.capture_exception(e)
    error_message = (
        f"{err['error_type']}: {err['error_message']}\n\n"
        f"{err['error_traceback']}"
    )

    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, 0, ended_timestamp, error_message
    )
    raise